# Day 47 — k-means and k-nearest neighbours

Two of the most-used algorithms in machine learning, and between them they contain about
six lines of logic:

```
k-means   : repeat { assign each point to the nearest centroid
                     move each centroid to the mean of its members }
k-nearest : for a query, take the k nearest training points and let them vote
```

Neither of those rules is where the bugs are. Both algorithms are a **distance function**
plus a rule, and every failure worth knowing about lives in the distance function - which
raises nothing, ever. It silently answers a different question than the one you asked.

This notebook walks through eight of those failures with the numbers attached: the
objective that can never tell you how many clusters there are, the random start that
changes the answer, the centroid that loses all of its points, the column whose units
decide the clustering, the update rule that does not match the distance it was paired
with, the shape assumption baked into squared distance, the k = 1 model that scores 1.000
on its own training data, and the extra columns that make "nearest" stop meaning anything.

## The distance matrix, and the one line that is not cosmetic

`|x - c|² = |x|² - 2 x·c + |c|²` turns a double loop into one matrix product. The
expansion subtracts two large numbers, so a point's distance to its own position can come
out as a tiny *negative* value - and `sqrt` of that is `nan`. The clamp is part of the
algorithm, not a tidy-up.

In [1]:
# ======================================================== 1. distances, once and for all
def sq_dists(X, C):
    """Squared euclidean distance from every row of X to every row of C.

    The expansion |x - c|^2 = |x|^2 - 2 x.c + |c|^2 turns a double loop into one
    matrix product (Day 44).  `np.maximum(..., 0)` is not cosmetic: the expansion
    subtracts two large numbers, so a point's distance to its own position comes
    out as a tiny negative value, and the sqrt of that is nan.
    """
    x2 = np.einsum('ij,ij->i', X, X)[:, None]
    c2 = np.einsum('ij,ij->i', C, C)[None, :]
    return np.maximum(x2 - 2.0 * X @ C.T + c2, 0.0)


def inertia(X, C, labels):
    """The objective k-means actually minimises: sum of squared distances to the
    assigned centroid.  Note the square - that is the whole story of section 6."""
    return float(np.sum((X - C[labels]) ** 2))


def l1_cost(X, C, labels):
    return float(np.sum(np.abs(X - C[labels])))


import math
import numpy as np
np.set_printoptions(precision=4, suppress=True)

X = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 2.0]])
C = np.array([[0.0, 0.0], [1.0, 2.0]])
print('squared distances from each point to each centroid:')
print(sq_dists(X, C))
print()
far = np.array([[1000000.5, 48000000.25]])
f2 = np.einsum('ij,ij->i', far, far)[:, None]
raw = (f2 - 2 * far @ far.T + f2)[0, 0]
print('a point against itself, via the expansion:', raw)
print('  sqrt of that                           :', np.sqrt(raw))
print('  with np.maximum(..., 0)                :', sq_dists(far, far)[0, 0])

squared distances from each point to each centroid:
[[0. 5.]
 [1. 4.]
 [4. 1.]]

a point against itself, via the expansion: -1.0
  sqrt of that                           : nan
  with np.maximum(..., 0)                : 0.0


## 1. Lloyd's algorithm

Assignment lowers the objective because every point picks its cheapest option. The update
lowers it because the mean is the minimiser of squared distance. A finite number of
possible partitions plus an objective that only goes down means the loop has to stop.
Hold on to *why* each half lowers it - that is the part that breaks later.

In [2]:
# ============================================================== 2. Lloyd's algorithm
def kmeans(X, k, seed=0, init='random', max_iter=100, on_empty='nan', trace=False):
    """Lloyd's algorithm.  The rule is two lines inside the loop.

    assign: every point joins its nearest centroid
    update: every centroid moves to the mean of its members

    Both halves lower the same objective - assignment because each point picks its
    cheapest option, update because the mean is the minimiser of squared distance -
    so the objective can only go down, and a finite number of partitions means it
    has to stop.  That proof is what breaks in section 6.
    """
    rng = np.random.default_rng(seed)
    C = kmeanspp_init(X, k, rng) if init == 'kmeans++' else X[rng.choice(len(X), k, replace=False)].copy()
    history = []
    labels = np.zeros(len(X), dtype=int)
    for it in range(max_iter):
        labels = np.argmin(sq_dists(X, C), axis=1)
        if trace:
            history.append((C.copy(), labels.copy(), inertia(X, C, labels)))
        newC = np.empty_like(C)
        for j in range(k):
            members = X[labels == j]
            if len(members) == 0:
                # The silent failure of section 5.  'nan' is what the naive mean does.
                newC[j] = np.nan if on_empty == 'nan' else X[np.argmax(np.min(sq_dists(X, C), axis=1))]
            else:
                newC[j] = members.mean(axis=0)
        if np.allclose(newC, C, equal_nan=True):
            C = newC
            break
        C = newC
    labels = np.argmin(sq_dists(X, C), axis=1)
    return C, labels, inertia(X, C, labels), it + 1, history


def kmeanspp_init(X, k, rng):
    """k-means++: pick the first centre uniformly, then pick each further centre with
    probability proportional to its squared distance to the nearest centre so far.

    It is Day 42's weighted random pick, and it is still random - it just makes the
    'two centres land in the same blob' draw unlikely instead of common.
    """
    C = [X[rng.integers(len(X))]]
    for _ in range(k - 1):
        d2 = np.min(sq_dists(X, np.array(C)), axis=1)
        C.append(X[rng.choice(len(X), p=d2 / d2.sum())])
    return np.array(C)


# =================================================================== 3. toy data sets
def blobs(centers, n_per, spread, seed=0):
    rng = np.random.default_rng(seed)
    cen = np.array(centers, dtype=float)
    return np.vstack([c + spread * rng.standard_normal((n_per, len(c))) for c in cen])


Xb = blobs([[0, 0], [6, 0.5], [3, 5]], 60, 0.9, seed=1)
C, lab, J, iters, hist = kmeans(Xb, 3, seed=0, init='kmeans++', trace=True)
print(f'180 points, three blobs, converged in {iters} iterations')
print('inertia per iteration: ' + ' -> '.join(f'{h[2]:.1f}' for h in hist))
print('monotone:', all(hist[i][2] >= hist[i + 1][2] - 1e-9 for i in range(len(hist) - 1)))
print('centroids:')
print(C[np.argsort(C[:, 0])])

180 points, three blobs, converged in 6 iterations
inertia per iteration: 1639.0 -> 1158.5 -> 1029.8 -> 447.9 -> 240.0 -> 239.0
monotone: True
centroids:
[[-0.0318 -0.0612]
 [ 2.8136  4.9725]
 [ 5.7976  0.4447]]


## 2. The objective can never tell you k

Inertia falls every time you add a centroid - at k = n it is exactly zero. So "minimise
the objective" is not a way to choose k, and the elbow is a judgement call about *where
the falling stops being worth it*, not a computation. Run the same sweep on 180 points of
uniform noise and the curve has the same shape with nothing to find.

In [3]:
rng = np.random.default_rng(11)
Xu = rng.random((180, 2)) * 6.0
print(f"{'k':>3} {'3 blobs':>12} {'drop':>8}   {'uniform noise':>14} {'drop':>8}")
pb = pu = None
for k in range(1, 9):
    Jb = kmeans(Xb, k, seed=0, init='kmeans++')[2]
    Ju = kmeans(Xu, k, seed=0, init='kmeans++')[2]
    db = '' if pb is None else f'{100 * (pb - Jb) / pb:6.1f}%'
    du = '' if pu is None else f'{100 * (pu - Ju) / pu:6.1f}%'
    print(f'{k:>3} {Jb:12.1f} {db:>8}   {Ju:14.1f} {du:>8}')
    pb, pu = Jb, Ju
print()
print('three tidy clusters of pure noise, sizes:', np.bincount(kmeans(Xu, 3, seed=0, init='kmeans++')[1]))

  k      3 blobs     drop    uniform noise     drop
  1       2180.5                    1092.2         
  2       1121.1    48.6%            667.4    38.9%
  3        239.0    78.7%            449.0    32.7%
  4        216.7     9.3%            269.4    40.0%
  5        202.2     6.7%            227.5    15.5%
  6        165.5    18.1%            195.6    14.0%
  7        132.7    19.8%            153.6    21.5%
  8        122.5     7.7%            140.8     8.3%

three tidy clusters of pure noise, sizes: [54 75 51]


## 3. The answer depends on the random start

Lloyd converges to *a* local minimum, not the global one. On three clearly separated
blobs - the easiest data set there is - a seed that drops two centroids into the same
blob splits that blob and merges the other two, and reports it without comment.

k-means++ is Day 42's weighted random pick: choose the first centre uniformly, then each
further centre with probability proportional to its squared distance to the nearest
centre chosen so far. It is still random; it just makes the bad draw rare.

In [4]:
Xs = blobs([[0, 0], [7, 0], [3.5, 6]], 60, 1.0, seed=2)
best = min(kmeans(Xs, 3, seed=s, init='kmeans++')[2] for s in range(200))
for init in ('random', 'kmeans++'):
    Js = [kmeans(Xs, 3, seed=s, init=init)[2] for s in range(200)]
    bad = sum(1 for j in Js if j > best * 1.01)
    print(f'{init:>9}: {bad:>3}/200 seeds end more than 1% above {best:.1f}; worst {max(Js):.1f}')
good = kmeans(Xs, 3, seed=0, init='random')
worst = max((kmeans(Xs, 3, seed=s, init='random') for s in range(200)), key=lambda r: r[2])
print()
print(f'seed 0     : inertia {good[2]:7.1f}  sizes {np.bincount(good[1])}')
print(f'worst seed : inertia {worst[2]:7.1f}  sizes {np.bincount(worst[1])}')

   random:  23/200 seeds end more than 1% above 367.0; worst 1742.1
 kmeans++:   1/200 seeds end more than 1% above 367.0; worst 1709.1

seed 0     : inertia   367.0  sizes [60 60 60]
worst seed : inertia  1742.1  sizes [ 21 120  39]


## 4. A centroid can lose all of its points

The mean of an empty cluster is `nan`, announced by one `RuntimeWarning` on stderr. What
happens next is the interesting part: `np.argmin` returns the index of a `nan` rather than
skipping it, so on the next round *every* point joins the dead centroid.

In [5]:
import warnings

Xe = np.array([[0.0, 0.0], [0.5, 0.45], [0.9, 0.05], [5.0, 5.0], [5.4, 4.6]])
Ce = np.array([[0.45, 0.17], [2.6, 2.6], [5.2, 4.8]])
la = np.argmin(sq_dists(Xe, Ce), axis=1)
print('assignment:', la.tolist(), '-> cluster 1 gets', int((la == 1).sum()), 'points')
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    empty = Xe[la == 1].mean(axis=0)
    print('mean of the empty cluster:', empty, ' warning:', str(w[0].message) if w else '-')
nan_c = np.array([Xe[la == j].mean(axis=0) if (la == j).any() else empty for j in range(3)])
print('np.argmin(' + str(np.round(sq_dists(Xe, nan_c)[0], 3).tolist()) + ') =',
      int(np.argmin(sq_dists(Xe, nan_c)[0])))
la2 = np.argmin(sq_dists(Xe, nan_c), axis=1)
print('next assignment:', la2.tolist(), '<- everything joined the dead centroid')
print('one more update:', [np.round(Xe[la2 == j].mean(axis=0), 3).tolist() if (la2 == j).any()
                           else [float('nan')] * 2 for j in range(3)])

assignment: [0, 0, 0, 2, 2] -> cluster 1 gets 0 points
mean of the empty cluster: [nan nan]  warning: Mean of empty slice.
np.argmin([0.246, nan, 50.08]) = 1
next assignment: [1, 1, 1, 1, 1] <- everything joined the dead centroid
one more update: [[nan, nan], [2.36, 2.02], [nan, nan]]


## 5. The units decide the clustering

360 requests with two columns: `prompt_tokens` (200 to 4000) and a cache hit rate (0 to
1). The three real populations differ *only* in hit rate. The token axis spans about
4000x more, so it contributes about 16 million times more squared distance, and raw
k-means cuts the data into three token bands.

Standardising is usually called the fix. It is not a fix, it is a different arbitrary
choice - "every column matters equally" - and here that still leaves a column of pure
noise voting on every distance.

In [6]:
def requests(n_per=120, seed=7):
    """n requests with two features in wildly different units:
    prompt_tokens (hundreds to thousands) and cache hit rate (0 to 1).

    The real structure is in the hit rate - three populations at 0.15, 0.50, 0.85 -
    while prompt length is drawn from the same wide range for all three.
    """
    rng = np.random.default_rng(seed)
    out, lab = [], []
    for j, hit in enumerate([0.15, 0.50, 0.85]):
        toks = rng.uniform(200, 4000, n_per)
        h = np.clip(hit + 0.05 * rng.standard_normal(n_per), 0, 1)
        out.append(np.column_stack([toks, h]))
        lab += [j] * n_per
    return np.vstack(out), np.array(lab)


def purity(labels, truth, k):
    """Fraction of points whose cluster's majority true label is their own."""
    ok = 0
    for j in range(k):
        m = labels == j
        if m.any():
            ok += np.bincount(truth[m]).max()
    return ok / len(truth)


Xr, yr = requests()
Cr, lr, Jr, _, _ = kmeans(Xr, 3, seed=0, init='kmeans++')
mu, sd = Xr.mean(axis=0), Xr.std(axis=0)
Cz, lz, _, _, _ = kmeans((Xr - mu) / sd, 3, seed=0, init='kmeans++')
Ch, lh, _, _, _ = kmeans(Xr[:, 1:2], 3, seed=0, init='kmeans++')
print(f'raw units     : purity {purity(lr, yr, 3):.3f}  centroid hit rates {np.sort(Cr[:, 1]).round(3)}')
print(f'standardised  : purity {purity(lz, yr, 3):.3f}  centroid hit rates '
      f'{np.sort(Cz[:, 1] * sd[1] + mu[1]).round(3)}')
print(f'hit rate only : purity {purity(lh, yr, 3):.3f}  centroid hit rates {np.sort(Ch[:, 0]).round(3)}')
print()
print(f'inertia of the raw-unit answer: {Jr:.0f} - a perfectly good number for the')
print('question it answered, which was not the question anyone asked.')

raw units     : purity 0.358  centroid hit rates [0.487 0.492 0.509]
standardised  : purity 0.669  centroid hit rates [0.302 0.311 0.828]
hit rate only : purity 1.000  centroid hit rates [0.143 0.492 0.854]

inertia of the raw-unit answer: 46140168 - a perfectly good number for the
question it answered, which was not the question anyone asked.


## 6. The update has to match the distance (Day 45, LeetCode 462)

Day 45 proved that the minimiser of `sum |x - c|` is the median, while the minimiser of
`sum (x - c)²` is the mean. That is not trivia: it is exactly the fact Lloyd's termination
proof rests on. Assign with L1 but update with the mean and the update step no longer
minimises anything - so the objective is free to go back up, and the proof is gone.

In [7]:
def kmedians(X, k, seed=0, max_iter=100, update='median'):
    """Same loop, L1 distance - and a switch for the update that is the point of
    section 6.  'median' is the correct partner for L1; 'mean' is the one that is
    copied over from k-means and quietly breaks the termination proof.
    """
    rng = np.random.default_rng(seed)
    C = X[rng.choice(len(X), k, replace=False)].copy().astype(float)
    costs = []
    for _ in range(max_iter):
        labels = np.argmin(np.abs(X[:, None, :] - C[None, :, :]).sum(axis=2), axis=1)
        costs.append(l1_cost(X, C, labels))
        newC = np.array([
            (np.median(X[labels == j], axis=0) if update == 'median' else X[labels == j].mean(axis=0))
            if np.any(labels == j) else C[j]
            for j in range(k)
        ])
        if np.allclose(newC, C):
            break
        C = newC
    labels = np.argmin(np.abs(X[:, None, :] - C[None, :, :]).sum(axis=2), axis=1)
    costs.append(l1_cost(X, C, labels))
    return C, labels, costs


def outlier_blobs(seed=0, n_per=30, n_out=6):
    """Three blobs plus a handful of points scattered far away - the shape of a
    real data set that has a few mislabelled or mis-parsed rows in it."""
    rng = np.random.default_rng(seed)
    return np.vstack([blobs([[0, 0], [6, 0], [3, 5]], n_per, 1.0, seed=seed),
                      rng.uniform(-6, 14, (n_out, 2))])


v = np.array([1.0, 2.0, 3.0, 100.0])
print(f'on {v.tolist()}: mean {v.mean():.1f} costs L1 {np.abs(v - v.mean()).sum():.1f}, '
      f'median {np.median(v):.1f} costs L1 {np.abs(v - np.median(v)).sum():.1f}')
Xo = outlier_blobs(seed=0)
c_mean = kmedians(Xo, 3, seed=6, update='mean')[2]
c_med = kmedians(Xo, 3, seed=6, update='median')[2]
print()
print('mean update   : ' + ' -> '.join(f'{c:.1f}' for c in c_mean))
print('median update : ' + ' -> '.join(f'{c:.1f}' for c in c_med))
up = next(i for i in range(len(c_mean) - 1) if c_mean[i + 1] > c_mean[i] + 1e-9)
print()
print(f'the mean version went UP at step {up}: {c_mean[up]:.2f} -> {c_mean[up + 1]:.2f}')
print(f'and settled {c_mean[-1] - c_med[-1]:.2f} higher ({c_mean[-1]:.2f} vs {c_med[-1]:.2f})')

on [1.0, 2.0, 3.0, 100.0]: mean 26.5 costs L1 147.0, median 2.5 costs L1 100.0

mean update   : 562.5 -> 340.2 -> 342.3 -> 329.4 -> 244.1 -> 196.6 -> 196.5 -> 196.5
median update : 562.5 -> 334.0 -> 323.3 -> 311.7 -> 240.7 -> 193.4 -> 192.4 -> 192.4

the mean version went UP at step 1: 340.18 -> 342.32
and settled 4.08 higher (196.53 vs 192.44)


## 7. Squared euclidean distance only knows one shape

Squared distance to one centre has circular level sets, and a point joins whichever
centre is nearest, so the border between two clusters is the perpendicular bisector of
the line joining them: every cluster k-means can express is a convex cell with straight
sides. Long bars get cut across the middle, moons get split down the middle - and a small
group next to a big one simply disappears, behind a healthy-looking score.

In [8]:
def elongated(n=150, seed=3):
    """Two groups that are long in x and thin in y, side by side in y."""
    rng = np.random.default_rng(seed)
    a = np.column_stack([rng.uniform(-4, 4, n), 0.9 + 0.22 * rng.standard_normal(n)])
    b = np.column_stack([rng.uniform(-4, 4, n), -0.9 + 0.22 * rng.standard_normal(n)])
    return np.vstack([a, b]), np.array([0] * n + [1] * n)


def moons(n=150, seed=4, noise=0.09):
    rng = np.random.default_rng(seed)
    t = np.linspace(0, math.pi, n)
    a = np.column_stack([np.cos(t), np.sin(t)]) + noise * rng.standard_normal((n, 2))
    b = np.column_stack([1 - np.cos(t), 0.5 - np.sin(t)]) + noise * rng.standard_normal((n, 2))
    return np.vstack([a, b]), np.array([0] * n + [1] * n)


for name, (Xd, yd) in [('two long bars', elongated()), ('two moons', moons())]:
    ld = kmeans(Xd, 2, seed=0, init='kmeans++')[1]
    print(f'{name:>14}: purity {purity(ld, yd, 2):.3f}  sizes {np.bincount(ld)}')
Xun = np.vstack([blobs([[0, 0]], 200, 1.0, seed=8), blobs([[2.2, 0]], 20, 0.5, seed=9)])
yun = np.array([0] * 200 + [1] * 20)
lun = kmeans(Xun, 2, seed=0, init='kmeans++')[1]
found = sum(1 for i in range(len(yun)) if yun[i] == 1
            and np.bincount(yun[lun == lun[i]]).argmax() == 1)
print(f"{'200 vs 20':>14}: purity {purity(lun, yun, 2):.3f}  sizes {np.bincount(lun)}")
print()
print(f'the 20-point group was recovered {found} times out of 20: k-means halved the big')
print('blob instead, and 0.909 is exactly the score for calling everything the majority.')

 two long bars: purity 0.523  sizes [145 155]
     two moons: purity 0.757  sizes [145 155]
     200 vs 20: purity 0.909  sizes [105 115]

the 20-point group was recovered 0 times out of 20: k-means halved the big
blob instead, and 0.909 is exactly the score for calling everything the majority.


## 8. k-nearest neighbours: the data is the model

There is no training step. The cost moved to query time, and `k` is the smoothing knob:
small k follows every piece of noise, k = n flattens the model into a constant.

The trap is the first column below. The nearest point to a training point is *itself*, so
k = 1 scores 1.000 on its own training set for any data set whose points are distinct.

In [9]:
# ======================================================================= 4. k-nearest
def knn_predict(Xtr, ytr, Xq, k, exclude_self=False):
    """The whole classifier.  There is no training: the data is the model."""
    d2 = sq_dists(Xq, Xtr)
    if exclude_self:
        # Writing the diagonal, not adding eye * inf: off the diagonal eye is 0,
        # and 0 * inf is nan, which would poison every other entry.
        d2 = d2.copy()
        np.fill_diagonal(d2, np.inf)
    nn = np.argsort(d2, axis=1, kind='stable')[:, :k]
    votes = ytr[nn]
    return np.array([np.bincount(v).argmax() for v in votes])


def accuracy(a, b):
    return float(np.mean(a == b))


Xk, yk = moons(120, seed=12, noise=0.22)
perm = np.random.default_rng(0).permutation(len(Xk))
tr, te = perm[:160], perm[160:]
print(f"{'k':>4} {'train acc':>10} {'train (self excluded)':>22} {'test acc':>10}")
for k in (1, 3, 5, 11, 25, 61, 160):
    a_in = accuracy(knn_predict(Xk[tr], yk[tr], Xk[tr], k), yk[tr])
    a_ex = accuracy(knn_predict(Xk[tr], yk[tr], Xk[tr], k, exclude_self=True), yk[tr])
    a_te = accuracy(knn_predict(Xk[tr], yk[tr], Xk[te], k), yk[te])
    print(f'{k:>4} {a_in:>10.3f} {a_ex:>22.3f} {a_te:>10.3f}')
print()
print('an even k can also tie, and numpy breaks ties by smallest index:')
print('  np.bincount([0, 1]).argmax() =', np.bincount(np.array([0, 1])).argmax(),
      '<- class 0 wins every tie in the data set')

   k  train acc  train (self excluded)   test acc
   1      1.000                  0.956      0.887
   3      0.981                  0.981      0.938
   5      0.988                  0.969      0.963
  11      0.975                  0.969      0.975
  25      0.956                  0.956      0.975
  61      0.875                  0.869      0.912
 160      0.500                  0.500      0.500

an even k can also tie, and numpy breaks ties by smallest index:
  np.bincount([0, 1]).argmax() = 0 <- class 0 wins every tie in the data set


## 9. In high dimensions nothing is near

Every extra column adds one more draw to an average, and averages concentrate. By a few
hundred dimensions the furthest of 500 random points is barely further away than the
nearest, so "nearest" has stopped carrying information.

The practical version is worse, because it does not look like a geometry problem: keep
exactly two informative columns and pad with noise.

In [10]:
def contrast(n, d, seed=0, queries=50):
    """How much nearer is the nearest point than the furthest one?

    (max - min) / min over uniform random points.  In two dimensions the nearest
    neighbour is genuinely near; by a few hundred dimensions every point is almost
    exactly the same distance away, and 'nearest' stops carrying information.
    """
    rng = np.random.default_rng(seed)
    X = rng.random((n, d))
    Q = rng.random((queries, d))
    dd = np.sqrt(sq_dists(Q, X))
    return float(np.mean((dd.max(axis=1) - dd.min(axis=1)) / dd.min(axis=1)))


def noisy_features(n=400, d=2, seed=5):
    """Two informative features, d - 2 pure noise ones.  Nothing about the problem
    changed; the only thing that grew is the number of dimensions the distance
    function has to average over."""
    rng = np.random.default_rng(seed)
    y = rng.integers(0, 2, n)
    X = np.column_stack([
        np.where(y == 1, 1.2, -1.2) + 0.9 * rng.standard_normal(n),
        np.where(y == 1, -0.8, 0.8) + 0.9 * rng.standard_normal(n),
    ])
    if d > 2:
        X = np.column_stack([X, rng.standard_normal((n, d - 2))])
    return X, y


print(f"{'dims':>6} {'(max-min)/min':>16}")
for d in (2, 4, 8, 16, 64, 256, 1024):
    print(f'{d:>6} {contrast(500, d, seed=0):>16.3f}')
print()
print(f"{'dims':>6} {'informative':>12} {'knn k=11 test acc':>20}")
for d in (2, 4, 10, 30, 100, 400):
    Xn, yn = noisy_features(400, d, seed=5)
    rn = np.random.default_rng(1).permutation(400)
    a = accuracy(knn_predict(Xn[rn[:300]], yn[rn[:300]], Xn[rn[300:]], 11), yn[rn[300:]])
    print(f'{d:>6} {2:>12} {a:>20.3f}')

  dims    (max-min)/min
     2           81.292
     4            9.940
     8            3.413
    16            1.520
    64            0.544
   256            0.231
  1024            0.110

  dims  informative    knn k=11 test acc
     2            2                0.940
     4            2                0.940
    10            2                0.920
    30            2                0.860
   100            2                0.790
   400            2                0.720


## 10. Making it cheaper: LeetCode 973 and LeetCode 658

Brute-force k-nearest is `O(n*d)` per query however it is written - vectorising buys a
constant, not a smaller exponent. There are exactly three ways out, and they fail in this
order:

- **one dimension**: the data can be sorted, so binary search it (LeetCode 658)
- **k out of n**: you need a *set*, not an order, so quickselect it (LeetCode 973, and
  Day 21's selection problem)
- **many dimensions**: no exact index survives, so approximate it and accept a recall
  below 1.0 as the price (the HNSW graph from Day 31)

### LeetCode 973 — K Closest Points to Origin

Two details worth keeping. Compare `x*x + y*y` rather than its square root: `sqrt` is
monotone so it cannot change the ordering, and on integer inputs it is also the difference
between an exact comparison and a floating-point one. And the answer is a set, so the
output order does not matter - which is precisely what lets you avoid a full sort.

In [11]:
# ======================================================== 5. LeetCode 973 and 658
class SolutionKClosest:
    def kClosest(self, points: list[list[int]], k: int) -> list[list[int]]:
        # A max-heap of size k: push, and drop the worst as soon as there are k + 1.
        # Comparing x*x + y*y instead of its square root is both faster and exact -
        # the square root is monotone, so it cannot change the ordering.
        heap: list[tuple[int, int, int]] = []
        for x, y in points:
            heapq.heappush(heap, (-(x * x + y * y), x, y))
            if len(heap) > k:
                heapq.heappop(heap)
        return [[x, y] for _, x, y in heap]


def k_closest_quickselect(points, k, seed=0):
    """Day 21's quickselect: partition around a random pivot, recurse into the side
    that still contains the boundary.  O(n) expected instead of O(n log k)."""
    pts = [tuple(p) for p in points]
    rng = random.Random(seed)

    def key(p):
        return p[0] * p[0] + p[1] * p[1]

    lo, hi = 0, len(pts) - 1
    arr = pts[:]
    while lo < hi:
        pivot = key(arr[rng.randint(lo, hi)])
        i, j = lo, hi
        while i <= j:
            while key(arr[i]) < pivot:
                i += 1
            while key(arr[j]) > pivot:
                j -= 1
            if i <= j:
                arr[i], arr[j] = arr[j], arr[i]
                i, j = i + 1, j - 1
        if k - 1 <= j:
            hi = j
        elif k - 1 >= i:
            lo = i
        else:
            break
    return [list(p) for p in arr[:k]]


import heapq, random
pts = [[3, 3], [5, -1], [-2, 4]]
print('points', pts, 'squared distances', [p[0] ** 2 + p[1] ** 2 for p in pts])
print('heap       ->', sorted(SolutionKClosest().kClosest(pts, 2)))
print('quickselect->', sorted(k_closest_quickselect(pts, 2)))
P = np.random.default_rng(21).integers(-10000, 10000, (4000, 2)).tolist()
ref = sorted(sorted(P, key=lambda p: p[0] ** 2 + p[1] ** 2)[:50])
print()
print('4,000 random points, k=50: heap matches', sorted(SolutionKClosest().kClosest(P, 50)) == ref,
      '| quickselect matches', sorted(k_closest_quickselect(P, 50, seed=1)) == ref)
big = [46341, 46341]
print('exact as ints:', big[0] ** 2 + big[1] ** 2, ' as a float distance:',
      (big[0] ** 2 + big[1] ** 2) ** 0.5)

points [[3, 3], [5, -1], [-2, 4]] squared distances [18, 26, 20]
heap       -> [[-2, 4], [3, 3]]
quickselect-> [[-2, 4], [3, 3]]

4,000 random points, k=50: heap matches True | quickselect matches True
exact as ints: 4294976562  as a float distance: 65536.0706939316


### LeetCode 658 — Find K Closest Elements

The k closest values to `x` in a *sorted* array are contiguous, so the answer is a window
and the search is over its left edge rather than over the values. Window `[lo, lo+k)`
beats `[lo+1, lo+k+1)` exactly when dropping `arr[lo]` costs more than gaining
`arr[lo+k]` saves.

`>` and not `>=` is the entire tie rule: when both candidates are equally far, the
smaller element wins. Swapping one character passes every short example and then differs
from the reference on a few percent of random inputs.

In [12]:
class SolutionFindClosestElements:
    def findClosestElements(self, arr: list[int], k: int, x: int) -> list[int]:
        # Binary search over the left edge of the window, not over the values.
        # The window [lo, lo + k) is better than [lo + 1, lo + k + 1) exactly when
        # dropping arr[lo] costs more than it saves by gaining arr[lo + k].
        # `>` and not `>=` is the tie rule: on a tie we keep the smaller element.
        lo, hi = 0, len(arr) - k
        while lo < hi:
            mid = (lo + hi) // 2
            if x - arr[mid] > arr[mid + k] - x:
                lo = mid + 1
            else:
                hi = mid
        return arr[lo:lo + k]


def closest_elements_by_abs(arr, k, x):
    """The sort-by-distance answer.  Correct, O(n log n), and it throws away the
    fact that the input was already sorted."""
    return sorted(sorted(arr, key=lambda v: (abs(v - x), v))[:k])


def closest_elements_ge(arr, k, x):
    """The same binary search with `>=` instead of `>`: the tie goes the wrong way."""
    lo, hi = 0, len(arr) - k
    while lo < hi:
        mid = (lo + hi) // 2
        if x - arr[mid] >= arr[mid + k] - x:
            lo = mid + 1
        else:
            hi = mid
    return arr[lo:lo + k]


print('arr=[1,2,3,4,5], k=4, x=3   ->', SolutionFindClosestElements().findClosestElements([1, 2, 3, 4, 5], 4, 3))
print('arr=[1,1,2,3,4,5], k=4, x=-1 ->', SolutionFindClosestElements().findClosestElements([1, 1, 2, 3, 4, 5], 4, -1))
print()
print('the tie, arr=[1,2,3,4,5], k=2, x=3 - both 2 and 4 are one away:')
print('  with `>`  (correct):', SolutionFindClosestElements().findClosestElements([1, 2, 3, 4, 5], 2, 3))
print('  with `>=`          :', closest_elements_ge([1, 2, 3, 4, 5], 2, 3))
rng658 = np.random.default_rng(31)
bad = 0
for _ in range(3000):
    n = int(rng658.integers(1, 40))
    a = sorted(int(v) for v in rng658.integers(-50, 50, n))
    kk = int(rng658.integers(1, n + 1))
    xx = int(rng658.integers(-60, 60))
    bad += closest_elements_ge(a, kk, xx) != closest_elements_by_abs(a, kk, xx)
print()
print(f'over 3,000 random cases the `>=` version differs {bad} times ({100 * bad / 3000:.1f}%)')

arr=[1,2,3,4,5], k=4, x=3   -> [1, 2, 3, 4]
arr=[1,1,2,3,4,5], k=4, x=-1 -> [1, 1, 2, 3]

the tie, arr=[1,2,3,4,5], k=2, x=3 - both 2 and 4 are one away:
  with `>`  (correct): [2, 3]
  with `>=`          : [3, 4]

over 3,000 random cases the `>=` version differs 104 times (3.5%)


## Tests

Everything above, re-derived from scratch and asserted.

In [13]:
def check() -> None:
    """Everything the walkthrough claims, re-derived from scratch."""
    Xb = blobs([[0, 0], [6, 0.5], [3, 5]], 60, 0.9, seed=1)
    # Lloyd never increases its own objective.
    _, _, _, _, h2 = kmeans(Xb, 3, seed=0, init='kmeans++', trace=True)
    assert all(h2[i][2] >= h2[i + 1][2] - 1e-9 for i in range(len(h2) - 1))
    # Adding a centroid can never raise the inertia.
    Js = [kmeans(Xb, k, seed=0, init='kmeans++')[2] for k in range(1, 7)]
    assert all(Js[i] >= Js[i + 1] - 1e-9 for i in range(len(Js) - 1))
    # k=1 inertia is the total spread around the global mean - no algorithm involved.
    assert abs(Js[0] - float(np.sum((Xb - Xb.mean(axis=0)) ** 2))) < 1e-6

    # Raw units lose the structure; standardising helps but does not recover it;
    # dropping the irrelevant column does.
    Xr, yr = requests()
    pr = purity(kmeans(Xr, 3, seed=0, init='kmeans++')[1], yr, 3)
    Xz = (Xr - Xr.mean(axis=0)) / Xr.std(axis=0)
    pz = purity(kmeans(Xz, 3, seed=0, init='kmeans++')[1], yr, 3)
    ph = purity(kmeans(Xr[:, 1:2], 3, seed=0, init='kmeans++')[1], yr, 3)
    assert ph == 1.0 and ph > pz > pr

    # The big blob is split in half and the small group is lost entirely.
    Xun = np.vstack([blobs([[0, 0]], 200, 1.0, seed=8), blobs([[2.2, 0]], 20, 0.5, seed=9)])
    yun = np.array([0] * 200 + [1] * 20)
    lun = kmeans(Xun, 2, seed=0, init='kmeans++')[1]
    found = sum(1 for i in range(len(yun)) if yun[i] == 1 and
                np.bincount(yun[lun == lun[i]]).argmax() == 1)
    assert found == 0 and purity(lun, yun, 2) == 200 / 220

    # The mean update breaks L1 monotonicity, and ends up worse.
    Xo = outlier_blobs(seed=0)
    c_mean = kmedians(Xo, 3, seed=6, update='mean')[2]
    c_med = kmedians(Xo, 3, seed=6, update='median')[2]
    assert any(c_mean[i + 1] > c_mean[i] + 1e-9 for i in range(len(c_mean) - 1))
    assert c_mean[-1] > c_med[-1]
    # The median is the L1 minimiser (Day 45, LeetCode 462).
    v = np.array([1.0, 2.0, 3.0, 100.0])
    assert np.abs(v - np.median(v)).sum() < np.abs(v - v.mean()).sum()

    # k=1 training accuracy is 1.0 whatever the data, unless you exclude the point itself.
    Xk, yk = moons(120, seed=12, noise=0.22)
    tr = np.random.default_rng(0).permutation(len(Xk))[:160]
    assert accuracy(knn_predict(Xk[tr], yk[tr], Xk[tr], 1), yk[tr]) == 1.0
    assert accuracy(knn_predict(Xk[tr], yk[tr], Xk[tr], 1, exclude_self=True), yk[tr]) < 1.0
    for s_ in range(5):
        Xn_, yn_ = noisy_features(120, 6, seed=s_)
        assert accuracy(knn_predict(Xn_, yn_, Xn_, 1), yn_) == 1.0

    # Distance concentration is monotone in the number of dimensions.
    cons = [contrast(500, d, seed=0) for d in (2, 4, 8, 16, 64, 256, 1024)]
    assert all(cons[i] > cons[i + 1] for i in range(len(cons) - 1))

    # The three k-closest routines agree, and squared distance orders points
    # identically to the distance itself.
    P = np.random.default_rng(21).integers(-10000, 10000, (4000, 2)).tolist()
    ref = sorted(sorted(P, key=lambda q: q[0] ** 2 + q[1] ** 2)[:50])
    assert sorted(SolutionKClosest().kClosest(P, 50)) == ref
    assert sorted(k_closest_quickselect(P, 50, seed=1)) == ref
    chk = np.random.default_rng(77).integers(-500, 500, (300, 2))
    assert np.array_equal(np.argsort((chk ** 2).sum(axis=1), kind='stable'),
                          np.argsort(np.sqrt((chk ** 2).sum(axis=1)), kind='stable'))

    # LeetCode 658: the window version matches sort-by-distance everywhere, and the
    # `>=` variant does not.
    rng = np.random.default_rng(32)
    diffs = 0
    for _ in range(3000):
        n = int(rng.integers(1, 40))
        a = sorted(int(x) for x in rng.integers(-50, 50, n))
        kk = int(rng.integers(1, n + 1))
        xx = int(rng.integers(-60, 60))
        ok = closest_elements_by_abs(a, kk, xx)
        assert SolutionFindClosestElements().findClosestElements(a, kk, xx) == ok
        diffs += closest_elements_ge(a, kk, xx) != ok
    assert diffs > 0
    assert SolutionFindClosestElements().findClosestElements([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
    assert SolutionFindClosestElements().findClosestElements([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]


check()
print('all assertions passed')

all assertions passed
